# Contrast Response Demo: Flash

Gratings have their own notebook (`3_contrast_grating_demo.ipynb`); spots are excluded for now (no example spot-contrast data yet).

Uses mean firing rate (`mean_rate`/`mean_rate_noise_sub`), not F1 -- a flash is a
single on/off step, not periodic, so F1 doesn't apply.

**Protocol name not fully confirmed** -- flash may not even vary by "contrast"
specifically (could be `intensity`, `lightAmplitude`, etc.). Check the printed
`epoch_parameters` keys and set `FLASH_CONDITION_KEYS` accordingly (see the setup
cell below).

`mean_rate_noise_sub`'s baseline is the per-cell average stimulus-window
`mean_rate` from this experiment's own `FLASH_CONDITION_KEYS[0]=0` trials (e.g. 0%
contrast), not each trial's own pre-stimulus window. If your data has no trials at
exactly 0 for whatever `FLASH_CONDITION_KEYS[0]` resolves to, `mean_rate_noise_sub`
comes out NaN for every cell (a warning prints saying so) rather than silently
reverting to a different convention.


In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Find contrast-response datasets

Broad substring search (`'contrast'`) -- deliberately not exact-match, since spot
and flash protocol names aren't confirmed yet. Check the printed
`protocol_name`s below and fill in each section's `_PROTOCOL_NAME` variable
accordingly.

In [ ]:
contrast_search = ra.get_datasets_from_protocol_names('contrast')
print(contrast_search['protocol_name'].unique())
ra.scrollable_dataframe(contrast_search)


## Choose an experiment (shared across all sections)

Only `exp_name` needs to be set. The white-noise/cell-typing chunk is found once
here (`ra.find_classified_noise_chunk`, same picker as the DS/OS demo -- prefers
a classified NDF0 chunk, falls back to NDF1, prints what it found) and reused by
every section below, since cell typing doesn't depend on which contrast stimulus
is being analyzed. Set `MANUAL_ANALYSIS_CHUNK` to override.

In [ ]:
exp_name = '20251222A'

MANUAL_ANALYSIS_CHUNK = None  # e.g. 'chunk13' or 'data001' -- set this to skip auto-detection

analysis_chunk_name = MANUAL_ANALYSIS_CHUNK or ra.find_classified_noise_chunk(exp_name)
if analysis_chunk_name is None:
    print(
        'No classified NDF0/NDF1 white-noise chunk found -- each section below will '
        'fall back to create_mea_pipeline\'s own nearest-chunk logic. This may crash; '
        'if so, set MANUAL_ANALYSIS_CHUNK above to a chunk you know is classified.'
    )


## Load cell typing (shared across all sections)

Builds an `AnalysisChunk` for the chunk found above just to read its
classification file(s) -- `b_load_spatial_maps=False`/`include_ei=False` since
none of this notebook needs RF maps or EIs, only the cell_id -> cell_type
mapping. If the chunk has more than one classification file, the first one found
is used by default; set `PREFERRED_TYPING_FILE` to pick a specific one.

Falls back to assuming no STA cropping (with a printed warning) if a chunk's
`.globals` file is missing the RTMP tag, and prints a one-time diagnostic (raw
label text + current vocabulary) if over half the cells come back "Unknown" --
usually means `assets/cell_types.csv` doesn't have this lab's real type strings yet.


In [ ]:
PREFERRED_TYPING_FILE = None  # e.g. 'chunk13.classificationYT.txt' -- set to pick a specific file

cell_type_map = pd.Series(dtype=object)

if analysis_chunk_name is not None:
    typing_chunk = ra.AnalysisChunk(
        exp_name, analysis_chunk_name, b_load_spatial_maps=False, include_ei=False, verbose=False,
    )
    classification_files = [f for f in typing_chunk.typing_files if 'classification' in f.lower()]
    if not classification_files:
        print(f'{analysis_chunk_name} has no classification file -- cell_type will be "Unknown" for every cell.')
    else:
        chosen_file = (
            PREFERRED_TYPING_FILE if PREFERRED_TYPING_FILE in classification_files
            else classification_files[0]
        )
        if len(classification_files) > 1:
            print(f'Multiple classification files found: {classification_files}. Using {chosen_file}.')
        file_idx = typing_chunk.typing_files.index(chosen_file)
        cell_type_map = pd.Series(
            typing_chunk.df_cell_params[f'typing_file_{file_idx}'].values,
            index=typing_chunk.cell_ids,
        )
        print(f'Loaded cell types for {len(cell_type_map)} cells from {chosen_file}.')
        print(cell_type_map.value_counts())

        if len(cell_type_map) > 0 and (cell_type_map == 'Unknown').mean() > 0.5:
            print()
            print('Over half of cells are "Unknown" -- likely a cell_types.csv vocabulary '
                  'mismatch, not a real absence of classifications.')
            typing_file_path = typing_chunk._resolve_typing_file_path(chosen_file)
            if typing_file_path is not None:
                print(f'Raw lines from {typing_file_path}:')
                with open(typing_file_path, 'r') as f:
                    raw_lines = [line.strip() for line in f if line.strip()][:5]
                for line in raw_lines:
                    print(f'  {line!r}')
            else:
                print(f'Could not resolve a file path for {chosen_file} to show raw content.')
            import importlib.resources as ir
            import retinanalysis
            cell_types_csv_path = str(ir.files(retinanalysis) / 'assets/cell_types.csv')
            with open(cell_types_csv_path, 'r') as f:
                csv_entries = [l.strip() for l in f if l.strip() and l.strip() != 'cell_types']
            print(f'Current assets/cell_types.csv vocabulary ({len(csv_entries)} entries): {csv_entries}')
else:
    print('No analysis_chunk_name -- skipping cell typing, every cell will show as "Unknown".')


## Shared response-extraction and plotting functions

Defined in `src/retinanalysis/utils/contrast_response_utils.py` and imported below; used
by all three sections.

- `load_contrast_section(...)`: finds the datafile for a given protocol (via
  `ra.find_datafile_for_protocol`), builds the pipeline (reusing the shared
  `analysis_chunk_name` from above, with a settable `corr_cutoff` EI-matching threshold),
  builds the tidy response table (via `ra.build_trial_response_table`), tags each row with
  its `cell_type` (from `pipeline.resp.df_spike_times`, via `MEAPipeline`'s own EI-based
  cross-chunk cell mapping), and prints `epoch_parameters` keys for verification.
- `plot_crf(...)`: raw vs. noise-subtracted (rows, `show_noise_sub=False` to skip the
  noise-subtracted row) x non-normalized vs. per-cell-normalized (cols) -- population mean
  +/- SEM vs. condition.
- `plot_raster_overview_by_cell_type(...)` / `plot_rasters_for_cell_type(...)`: one figure
  PER cell type (not one combined grid) with a representative sample of cells, and every
  cell of a chosen type, paginated.

**Cell-type bookkeeping labels:** `'Unmatched'` means a cell never EI-matched the reference
classification chunk at all; `'Unknown'` means it matched but its classification label was
blank/unrecognized. Both are excluded from cell-type lists by default -- if most cells come
out "Unknown", check whether `assets/cell_types.csv` has your lab's actual mouse cell-type
vocabulary (see the typing cell above).


In [ ]:
from retinanalysis.utils.contrast_response_utils import *


---
# Section: Flash

**Protocol name not confirmed from here either** -- same situation as spots, and
flash may not even vary by "contrast" specifically (could be `intensity`,
`lightAmplitude`, etc.) -- check the printed `epoch_parameters` keys below and set
`FLASH_CONDITION_KEYS` accordingly. Uses mean firing rate, same reasoning as
spots above (a flash is a single on/off step, not periodic -- F1 doesn't apply).

In [ ]:
FLASH_PROTOCOL_NAME = 'manookinlab.protocols.ContrastResponseFlash'  # <-- VERIFY/REPLACE from contrast_search above
FLASH_CONDITION_KEYS = ['contrast']  # <-- VERIFY against the printed epoch_parameters keys below -- may need to change e.g. to ['intensity']
MANUAL_DATAFILE_NAME_FLASH = None  # e.g. 'data008' -- set to skip auto-detection
CORR_THRESHOLD_FLASH = 0.85  # EI-matching cutoff (reference chunk <-> this datafile)

df_trials_flash = spike_times_by_cell_flash = df_epochs_flash = datafile_name_flash = ndf_flash = None

if FLASH_PROTOCOL_NAME not in contrast_search['protocol_name'].unique():
    print(
        f'{FLASH_PROTOCOL_NAME!r} was not found in contrast_search -- this is a placeholder, '
        'not a confirmed protocol name. Update FLASH_PROTOCOL_NAME to one of: '
        f'{sorted(contrast_search["protocol_name"].unique())}. '
        'Skipping the rest of the Flash section until this is fixed.'
    )
else:
    # Loading prints collapse into a scrollable box.
    with scrollable_prints():
        df_trials_flash, spike_times_by_cell_flash, df_epochs_flash, datafile_name_flash, ndf_flash = load_contrast_section(
            exp_name, contrast_search, FLASH_PROTOCOL_NAME, condition_keys=FLASH_CONDITION_KEYS,
            analysis_chunk_name=analysis_chunk_name, corr_cutoff=CORR_THRESHOLD_FLASH,
            manual_datafile_name=MANUAL_DATAFILE_NAME_FLASH, typing_chunk=typing_chunk,
            baseline_condition_key=FLASH_CONDITION_KEYS[0], baseline_condition_value=0.0,
        )


## Flash response curve (mean rate) and rasters

Population curve (raw + noise-subtracted, non-normalized + per-cell-normalized), then raster overviews, for the Flash section.


In [ ]:
if df_trials_flash is None:
    print('Flash section not runnable yet (see the cell above) -- skipping this plot.')
else:
    fig = plot_crf(
        df_trials_flash, condition_key=FLASH_CONDITION_KEYS[0],
        response_col='mean_rate_noise_sub', raw_response_col='mean_rate',
        title=f'Flash: mean rate vs. {FLASH_CONDITION_KEYS[0]} (NDF {ndf_flash})',
        log_x=False, even_spacing=True,
    )


Same plot, true-to-scale log-x axis instead of the evenly-spaced linear one.

In [ ]:
if df_trials_flash is None:
    print('Flash section not runnable yet (see the cell above) -- skipping this plot.')
else:
    fig = plot_crf(
        df_trials_flash, condition_key=FLASH_CONDITION_KEYS[0],
        response_col='mean_rate_noise_sub', raw_response_col='mean_rate',
        title=f'Flash: mean rate vs. {FLASH_CONDITION_KEYS[0]} (NDF {ndf_flash}) -- log scale',
        log_x=True,
    )


In [ ]:
if df_trials_flash is None:
    print('Flash section not runnable yet (see the cell above) -- skipping this plot.')
else:
    raster_figs_flash = plot_raster_overview_by_cell_type(
        df_trials_flash, spike_times_by_cell_flash, df_epochs_flash,
        condition_key=FLASH_CONDITION_KEYS[0], response_col='mean_rate_noise_sub',
        markersize=2.2,  # flash-only, a bit thicker than the shared default of 1.5.
    )


In [ ]:
if df_trials_flash is None:
    print('Flash section not runnable yet (see the cell above) -- skipping this plot.')
else:
    _real_types_flash = sorted(
        t for t in df_trials_flash['cell_type'].unique() if t not in ('Unknown', 'Unmatched')
    )
    SELECTED_CELL_TYPE_FLASH = (
        _real_types_flash[0] if _real_types_flash
        else sorted(df_trials_flash['cell_type'].unique())[0]
    )  # change to any type printed above
    print(f'Selected cell type: {SELECTED_CELL_TYPE_FLASH!r} (available: {sorted(df_trials_flash["cell_type"].unique())})')

    raster_figs_flash_selected = plot_rasters_for_cell_type(
        df_trials_flash, spike_times_by_cell_flash, df_epochs_flash,
        condition_key=FLASH_CONDITION_KEYS[0], selected_cell_type=SELECTED_CELL_TYPE_FLASH,
        markersize=2.2,  # flash-only, a little thicker than the shared default (1.5).
    )


### Optional: look at a different NDF (light level) or cell type

Same idea as the grating section's explorer -- reruns this section's rasters at a
different light level via `ra.get_ndf_blocks_for_protocol`. Leave
`EXPLORE_NDF_FLASH = None` to skip this. Only runs if the Flash section above found
real data (FLASH_PROTOCOL_NAME confirmed).

In [ ]:
EXPLORE_NDF_FLASH = None  # e.g. 2.0 -- set to an NDF value to look at a different light level
EXPLORE_CELL_TYPE_FLASH = None  # e.g. 'off brisk sustained' -- None reuses SELECTED_CELL_TYPE_FLASH

if df_trials_flash is None:
    print('Flash section not runnable yet -- skipping the NDF explorer too.')
elif EXPLORE_NDF_FLASH is not None:
    df_ndf_blocks_flash = ra.get_ndf_blocks_for_protocol(exp_name, FLASH_PROTOCOL_NAME)
    match = df_ndf_blocks_flash[df_ndf_blocks_flash['NDF'] == EXPLORE_NDF_FLASH]
    if len(match) == 0:
        print(f'No {FLASH_PROTOCOL_NAME} block found at NDF {EXPLORE_NDF_FLASH} for {exp_name}. '
              f'Available NDFs: {list(df_ndf_blocks_flash["NDF"])}')
    else:
        explore_datafile = match.iloc[0]['datafile_name']
        print(f'Loading NDF {EXPLORE_NDF_FLASH} ({explore_datafile}) ...')
        with scrollable_prints():
            df_trials_explore, spike_times_by_cell_explore, df_epochs_explore, _, _ = load_contrast_section(
                exp_name, contrast_search, FLASH_PROTOCOL_NAME, condition_keys=FLASH_CONDITION_KEYS,
                analysis_chunk_name=analysis_chunk_name, corr_cutoff=CORR_THRESHOLD_FLASH,
                manual_datafile_name=explore_datafile,
                baseline_condition_key=FLASH_CONDITION_KEYS[0], baseline_condition_value=0.0,
            )
        explore_cell_type = EXPLORE_CELL_TYPE_FLASH or SELECTED_CELL_TYPE_FLASH
        raster_figs_explore_flash = plot_rasters_for_cell_type(
            df_trials_explore, spike_times_by_cell_explore, df_epochs_explore,
            condition_key=FLASH_CONDITION_KEYS[0], selected_cell_type=explore_cell_type,
            markersize=2.2,  # matches the rest of the flash section's slightly-thicker points.
        )
else:
    print('EXPLORE_NDF_FLASH is None -- skipping. Set it to an NDF value to explore a different light level.')
